### 6 · Multi-Turn Memory with LangGraph

Real users ask follow-up questions. Without memory, each query is isolated.
This notebook adds **conversation memory** to the RAG graph so it handles
multi-turn dialogue naturally.

### What you'll learn
```
1. LangGraph checkpointers — persist state across turns
2. Question condensation — rewrite follow-ups into standalone queries
3. Thread-based conversations — multiple users, independent histories
4. When to retrieve vs when to use memory alone
```

### Graph
```
START → condense_question → route_query → retrieve_docs → generate_answer → END
         (rewrites follow-ups)                                    ↑
                                                      (chat history injected)
```

In [ ]:
from pathlib import Path
from typing import TypedDict

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv

load_dotenv(override=True)

VECTORSTORE_DIR = Path("../data/vectorstore")
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = FAISS.load_local(
    str(VECTORSTORE_DIR), embeddings, allow_dangerous_deserialization=True
)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

RETRIEVERS = {
    "admin_tech_docs": {
        "retriever": vectorstore.as_retriever(search_kwargs={"k": 5}),
        "description": "Technical documentation — architecture, features, code quality, operations.",
    },
}

print(f"Loaded {vectorstore.index.ntotal} vectors. Memory notebook ready.")

---
### Step 1 — Understanding the Problem

Without memory, this happens:
```
User: How does the journey canvas work?
Bot:  The journey canvas uses React Flow to render...

User: What about its performance?
Bot:  ❌ I don't have information about performance.  ("its" = ???)
```

The retriever searches for "What about its performance?" — which is meaningless
without knowing "its" refers to "the journey canvas".

**Solution:** Before retrieval, rewrite the question using chat history:
- "What about its performance?" → "What is the performance of the journey canvas?"

---
### Step 2 — Define State & Nodes

State now includes `chat_history` — a list of previous messages.

In [ ]:
class GraphState(TypedDict):
    """State that flows through the memory-enabled RAG graph."""
    question: str                          # user's raw input
    standalone_question: str               # rewritten question (after condensation)
    chat_history: list                     # list of HumanMessage / AIMessage
    retriever_name: str                    # which retriever to use
    documents: list[Document]              # retrieved chunks
    answer: str                            # final generated answer

print("State schema defined with chat_history.")

In [ ]:
# ── Node 1: Condense Question ────────────────────────────────────────────────
# Rewrites follow-up questions into standalone queries using chat history.
# If no history, passes the question through unchanged.

CONDENSE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Given the following conversation history and a follow-up question, "
     "rewrite the follow-up question as a STANDALONE question that captures "
     "the full context. If the question is already standalone, return it as-is.\n\n"
     "IMPORTANT: Output ONLY the rewritten question, nothing else."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

def condense_question(state: GraphState) -> dict:
    """Rewrite follow-up questions into standalone queries."""
    history = state.get("chat_history", [])

    if not history:
        # First message — no rewriting needed
        print(f"  [condense] no history, passing through")
        return {"standalone_question": state["question"]}

    chain = CONDENSE_PROMPT | llm | StrOutputParser()
    standalone = chain.invoke({
        "chat_history": history,
        "question": state["question"],
    })
    print(f"  [condense] '{state['question']}' → '{standalone}'")
    return {"standalone_question": standalone}


# ── Node 2: Route Query ──────────────────────────────────────────────────────

ROUTER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query router. Given a user question, decide which knowledge base "
     "to search. Respond with ONLY the retriever name.\n\n"
     "Available retrievers:\n{retriever_list}\n\n"
     "Default: admin_tech_docs"),
    ("human", "{question}"),
])

def route_query(state: GraphState) -> dict:
    """Pick retriever based on the standalone question."""
    retriever_list = "\n".join(
        f"- {name}: {info['description']}" for name, info in RETRIEVERS.items()
    )
    chain = ROUTER_PROMPT | llm | StrOutputParser()
    chosen = chain.invoke({
        "question": state["standalone_question"],
        "retriever_list": retriever_list,
    }).strip()

    if chosen not in RETRIEVERS:
        chosen = "admin_tech_docs"

    print(f"  [router] → {chosen}")
    return {"retriever_name": chosen}


# ── Node 3: Retrieve Docs ────────────────────────────────────────────────────

def retrieve_docs(state: GraphState) -> dict:
    """Fetch docs using the standalone (rewritten) question."""
    retriever = RETRIEVERS[state["retriever_name"]]["retriever"]
    docs = retriever.invoke(state["standalone_question"])
    print(f"  [retrieve] → {len(docs)} chunks")
    return {"documents": docs}


# ── Node 4: Generate Answer ──────────────────────────────────────────────────

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant for the AMS Admin Tool team. "
     "Use the context below to answer the user's question. "
     "If the context doesn't contain enough info, say so.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

def generate_answer(state: GraphState) -> dict:
    """Generate answer with chat history for conversational tone."""
    context = "\n\n---\n\n".join(
        f"[{d.metadata.get('source', '?')}]\n{d.page_content}"
        for d in state["documents"]
    )
    chain = ANSWER_PROMPT | llm | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "chat_history": state.get("chat_history", []),
        "question": state["question"],
    })

    # Update chat history with this turn
    new_history = list(state.get("chat_history", []))
    new_history.append(HumanMessage(content=state["question"]))
    new_history.append(AIMessage(content=answer))

    print(f"  [generate] → {len(answer)} chars")
    return {"answer": answer, "chat_history": new_history}


print("Nodes defined: condense_question, route_query, retrieve_docs, generate_answer")

---
### Step 3 — Build the Graph with Checkpointer

The **checkpointer** (MemorySaver) persists graph state between invocations.
Each conversation gets a unique `thread_id` — like a session ID.

```
START → condense_question → route_query → retrieve_docs → generate_answer → END
```

In [ ]:
workflow = StateGraph(GraphState)

workflow.add_node("condense_question", condense_question)
workflow.add_node("route_query", route_query)
workflow.add_node("retrieve_docs", retrieve_docs)
workflow.add_node("generate_answer", generate_answer)

workflow.set_entry_point("condense_question")
workflow.add_edge("condense_question", "route_query")
workflow.add_edge("route_query", "retrieve_docs")
workflow.add_edge("retrieve_docs", "generate_answer")
workflow.add_edge("generate_answer", END)

# MemorySaver keeps state in-memory (use SqliteSaver for persistence across restarts)
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)

print("Graph compiled with MemorySaver checkpointer.")

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

---
### Step 4 — Helper for Chat

A simple `chat()` function that sends a message and prints the response.
The `thread_id` groups messages into the same conversation.

In [ ]:
def chat(question: str, thread_id: str = "default") -> str:
    """Send a message to the RAG graph and return the answer."""
    config = {"configurable": {"thread_id": thread_id}}

    # Get existing history from checkpoint
    state = graph.get_state(config)
    history = state.values.get("chat_history", []) if state.values else []

    result = graph.invoke(
        {"question": question, "chat_history": history},
        config=config,
    )
    return result["answer"]


print("chat() helper ready. Usage: chat('your question', thread_id='session-1')")

---
### Step 5 — Test Multi-Turn Conversation

Watch how the condenser rewrites follow-up questions using history.

In [ ]:
# ── Turn 1: Initial question ──────────────────────────────────────────────────

THREAD = "demo-session-1"

print("USER: How does the journey canvas work?")
print("=" * 70)
answer = chat("How does the journey canvas work?", thread_id=THREAD)
print(f"\nBOT: {answer}")

In [ ]:
# ── Turn 2: Follow-up (uses "it" — needs context from history) ─────────────

print("USER: What libraries does it use?")
print("=" * 70)
answer = chat("What libraries does it use?", thread_id=THREAD)
print(f"\nBOT: {answer}")

In [ ]:
# ── Turn 3: Another follow-up ─────────────────────────────────────────────────

print("USER: How is state managed there?")
print("=" * 70)
answer = chat("How is state managed there?", thread_id=THREAD)
print(f"\nBOT: {answer}")

In [ ]:
# ── Turn 4: Topic shift ───────────────────────────────────────────────────────

print("USER: Now tell me about code quality standards")
print("=" * 70)
answer = chat("Now tell me about code quality standards", thread_id=THREAD)
print(f"\nBOT: {answer}")

---
### Step 6 — Thread Isolation

Different `thread_id` = different conversation. No cross-talk.

In [ ]:
# A completely separate conversation

print("THREAD B — new user, no history")
print("=" * 70)
print("USER: What are the anti-patterns?")
answer = chat("What are the anti-patterns?", thread_id="user-B")
print(f"\nBOT: {answer}")

print("\n" + "=" * 70)
print("USER: Which one is the most dangerous?")
answer = chat("Which one is the most dangerous?", thread_id="user-B")
print(f"\nBOT: {answer}")

---
### Step 7 — Inspect Chat History

Peek inside the checkpointed state to see the accumulated history.

In [ ]:
# Inspect history for thread "demo-session-1"

config = {"configurable": {"thread_id": THREAD}}
state = graph.get_state(config)
history = state.values.get("chat_history", [])

print(f"Thread '{THREAD}' — {len(history)} messages:\n")
for msg in history:
    role = "USER" if isinstance(msg, HumanMessage) else "BOT"
    print(f"  [{role}] {msg.content[:120]}{'...' if len(msg.content) > 120 else ''}")
    print()

---
### Takeaways

| Concept | What you learned |
|---------|------------------|
| **Question condensation** | Rewrite follow-ups into standalone queries before retrieval |
| **MemorySaver** | In-memory checkpointer for LangGraph — state persists across turns |
| **Thread isolation** | Each `thread_id` is an independent conversation |
| **History in prompts** | Pass `chat_history` to the answer prompt for conversational tone |

**Production upgrades:**
- Swap `MemorySaver` for `SqliteSaver` or `PostgresSaver` for persistence across restarts
- Add a **history trimming** strategy (keep last N turns or summarize old turns)
- Add a `should_retrieve` node that skips retrieval for greetings/chitchat